# Hour 2 · Text similarity and clustering


**Goal:** compare tablets and see whether the machine rediscovers the philologists' genres.

**Needs:** `pip install scikit-learn plotly`; `umap-learn` gives the nicest map when available.

*Reading:* `docs/04-genres.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first ===
# Loads the Copenhagen Ugaritic Corpus (CUC) from the HuggingFace JSONL cache. No edits needed.
import os
import sys
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")  # keep sklearn quiet on some laptops
sys.path.append("..")                       # so Python can find data/loader.py
from data.loader import load_texts

texts = load_texts()                        # 278 real KTU tablets; cached after first download
print(f"Loaded {len(texts)} tablets.")
texts[0]["ktu"], texts[0]["genre"], texts[0]["lines"][:2]

## 1. Vectorize a genre-diverse set

In [ ]:
sample = [t for t in texts if t["genre"] in {"myth","letter","ritual","divination"}
          and len(t["tokens"]) >= 30]
genres = [t["genre"] for t in sample]
from data.loader import corpus_as_documents
labels, docs = corpus_as_documents(sample)
from sklearn.feature_extraction.text import TfidfVectorizer
X = TfidfVectorizer(token_pattern=r"[^\s]+").fit_transform(docs)
print(X.shape)

## 2. Nearest neighbours of one tablet

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
S = pd.DataFrame(cosine_similarity(X), index=labels, columns=labels)
TARGET = "2.10"   # a letter — TODO try a myth like "1.4"
print(S[TARGET].drop(TARGET).sort_values(ascending=False).head())

## 3. Cluster

In [ ]:
from sklearn.cluster import KMeans
k = 4
km = KMeans(n_clusters=k, random_state=0, n_init=10).fit(X)
for c in range(k):
    members = [labels[j] for j in range(len(labels)) if km.labels_[j]==c]
    print(f"cluster {c}: {members}")

## 4. Map the tablets interactively (UMAP, with PCA fallback)

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd
import warnings

try:
    warnings.filterwarnings("ignore", message="FNV hashing is not implemented in Numba.*")
    warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")
    import umap
    reducer = umap.UMAP(
        n_components=2,
        metric="cosine",
        n_neighbors=min(15, max(2, len(sample) - 1)),
        min_dist=0.12,
        random_state=0,
    )
    xy = reducer.fit_transform(X)
    method = "UMAP"
except Exception as exc:
    print(f"UMAP unavailable ({type(exc).__name__}: {exc}); using PCA instead.")
    xy = PCA(n_components=2, random_state=0).fit_transform(X.toarray())
    method = "PCA fallback"

plot_df = pd.DataFrame({
    "KTU": labels,
    "Genre": genres,
    "Tokens": [len(t["tokens"]) for t in sample],
    "Preview": [" / ".join(line for line in t["lines"] if line.strip())[:320] for t in sample],
    "x": xy[:, 0],
    "y": xy[:, 1],
})
plot_df["Cluster"] = [f"cluster {c}" for c in km.labels_] if "km" in globals() else "run cell 3"

try:
    import plotly.express as px
    fig = px.scatter(
        plot_df,
        x="x",
        y="y",
        color="Genre",
        symbol="Genre",
        hover_name="KTU",
        hover_data={"x": ":.3f", "y": ":.3f", "Tokens": True, "Cluster": True, "Preview": True},
        title=f"Tablets in TF-IDF space ({method})",
        width=950,
        height=650,
    )
    fig.update_traces(marker={"size": 10, "line": {"width": 0.5, "color": "white"}})
    fig.update_layout(legend_title_text="Genre", xaxis_title="dimension 1", yaxis_title="dimension 2")
    fig.show()
except Exception as exc:
    print(f"Plotly unavailable ({type(exc).__name__}: {exc}); showing a static fallback.")
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 6))
    for genre in sorted(plot_df["Genre"].unique()):
        rows = plot_df[plot_df["Genre"] == genre]
        plt.scatter(rows["x"], rows["y"], label=genre, s=60)
    for _, row in plot_df.iterrows():
        plt.annotate(row["KTU"], (row["x"], row["y"]), fontsize=7)
    plt.legend()
    plt.title(f"Tablets in TF-IDF space ({method})")
    plt.show()

## 5. Discussion and your turn
- Do letters cluster together? Do myths separate from rituals?
- Where does the machine agree with philologists, and where not?

**Your turn:** change `TARGET` above from `"2.10"` to a myth such as `"1.4"`, rerun the nearest-neighbour cell, then hover over nearby points on the map. Do the neighbours make philological sense?

> **Production version:** UgaritLab uses **gensim** + **UMAP** (`build_semantic_index.py`, `generate_stats.py`) and a **FastText** model (`semantic_search.py`) for similarity that goes beyond shared word-forms to sub-word/semantic similarity.